In [ ]:
# Paths come from the memtune package (see docs/INSTALL.md): no machine-specific locations.
#   data   -> data/lbfgs_runs.csv   (override: MEMTUNE_DATA_DIR)
#   models -> models/              (override: MEMTUNE_MODELS_DIR)
#   output -> results/             (override: MEMTUNE_RESULTS_DIR)
from memtune import paths
(paths.results_dir() / "retrained_models").mkdir(parents=True, exist_ok=True)  # notebooks never overwrite models/


### Data Preprocessing

In [ ]:
import pandas as pd
import numpy as np
from sklearn.multioutput import MultiOutputRegressor
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.ensemble import RandomForestRegressor
pd.set_option('display.max_columns', 200)
pd.set_option('display.max_colwidth', 20)
from sklearn.model_selection import RandomizedSearchCV
import numpy as np
from scipy.stats import randint, uniform
import joblib
import csv
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
import seaborn as sns

In [ ]:
df = pd.read_csv(str(paths.dataset_path()))
first = ["name", "nvar"]
df = df[first + [c for c in df.columns if c not in first]]
df["nvar"].value_counts
df = df.sort_values(["name", "nvar", "mem"]).reset_index()
df

In [ ]:
feature_cols_reg = [
                    # Core Features
                    "nvar", 
                    "mem",
                    "tree_length", 
                    "tree_depth", 
                    "time_obj",  # initial eval
                    "time_grad",
                    "extract_duration_problem",
                    
                    # Expression Tree Features
                    "highest_degree (ExprTree)",
                    "count_plus_minus",
                    "count_trigonometric_function",
                    "count_exponential_function",
                    "count_op_function",

                    # allocation
                    "alloc_obj",
                    "alloc_grad",
                    "total_alloc",
                  ]

target_cols_reg = [ "neval_obj",
                    "neval_grad",
                    "timed_bytes"] #(neval_obj, neval_grad, timed_bytes)

target_col_model = ["stats_elapsed_time"]

group_key = ["name", "nvar"]                                                                                                                   # [...]     
all_cols = list(set(feature_cols_reg + target_cols_reg + target_col_model + group_key))
solver_metrics = ['neval_grad', 'neval_obj', 'timed_bytes']
df[solver_metrics] = df[solver_metrics].fillna(0)
df = df[all_cols]
df.isna().sum().sort_values(ascending=False)

In [ ]:
df = df.copy()
df["mem_count"] = df.groupby(["name", "nvar"]).transform('size')
df = df[df["mem_count"] == 100].copy().reset_index(drop=True)
df

In [ ]:
# unique instances: one row per (problem, nvar)
instances = df[group_key].drop_duplicates()

# shuffle instances
instances = instances.sample(frac=1, random_state=66).reset_index(drop=True)

n = len(instances)
n_train = int(0.8 * n)

train_inst = instances.iloc[:n_train]
test_inst  = instances.iloc[n_train :]

# assign rows to splits by (problem, nvar)
train_df = df.merge(train_inst, on=["nvar", "name"], how="inner").reset_index(drop=True)
test_df  = df.merge(test_inst,  on=["nvar", "name"], how="inner").reset_index(drop=True)
all_df = pd.concat([train_df, test_df], axis=0)
all_df = all_df.reset_index(drop=True)
all_df

In [ ]:
X_train = train_df[feature_cols_reg].to_numpy(dtype=float)
X_test  = test_df[feature_cols_reg].to_numpy(dtype=float)

y_train = np.log1p(train_df[target_cols_reg].to_numpy(dtype=float))
y_test  = np.log1p(test_df[target_cols_reg].to_numpy(dtype=float))

X_all = np.concatenate([X_train, X_test], axis=0)


### Stage 1

In [ ]:
stage_1 = joblib.load(str(paths.models_dir() / "stage1/random_forest.pkl"))
# best_model = random_search.best_estimator_
rf_importances = stage_1.feature_importances_
indices_rf = np.argsort(rf_importances)[::-1]  

plt.figure(figsize=(10, 5))
plt.bar(range(len(feature_cols_reg)), rf_importances[indices_rf])
plt.xticks(range(len(feature_cols_reg)), [feature_cols_reg[i] for i in indices_rf], rotation=45, ha="right")
plt.ylabel("Feature importance")
plt.title("Stage1 Feature Importances")
plt.tight_layout()
plt.show()

pred = stage_1.predict(X_test)
test_mse = mean_squared_error(y_test, pred)
test_r2  = r2_score(y_test, pred)
print(f"Random Forest Stage 1: MSE={test_mse:.4f}  R²={test_r2:.4f}")

### Pipeline 2

In [ ]:
X_train_time = stage_1.predict(X_train)
X_test_time  = stage_1.predict(X_test)
y_train_time = np.log1p(train_df[target_col_model].to_numpy(dtype=float)).ravel()
y_test_time  = np.log1p(test_df[target_col_model].to_numpy(dtype=float)).ravel()

In [ ]:
stage2 = joblib.load(str(paths.models_dir() / "legacy/stage2_xgb_on_predicted_stage1.pkl"))
pred = stage2.predict(X_test_time)
test_mse = mean_squared_error(y_test_time, pred)
test_r2  = r2_score(y_test_time, pred)
print(f"Gradient Boosting Stage2: MSE={test_mse:.4f}  R\u00b2={test_r2:.4f}")

In [ ]:
xgb_importances_time = stage2.feature_importances_
indices_xgb = np.argsort(xgb_importances_time)[::-1]

plt.figure(figsize=(10, 5))
plt.bar(range(len(target_cols_reg)), xgb_importances_time[indices_xgb])
plt.xticks(range(len(target_cols_reg)), [target_cols_reg[i] for i in indices_xgb], rotation=45, ha="right")
plt.ylabel("Feature importance")
plt.title("Stage 2 Feature Importances")
plt.tight_layout()
plt.show()

### Final Mem Selection

In [ ]:
test_df["predicted_time"] = stage2.predict(X_test_time)
test_df["predicted_time"]

In [ ]:
group_cols = ["name", "nvar"]
best_idx = df.groupby(group_cols)["stats_elapsed_time"].idxmin()
best_mem_map = df.loc[best_idx].set_index(group_cols)["mem"]
df["best_observed_mem"] = df.set_index(group_cols).index.map(best_mem_map)

In [ ]:
group_cols = ["name", "nvar"]
best_idx = test_df.groupby(group_cols)["stats_elapsed_time"].idxmin()
best_mem_map = test_df.loc[best_idx].set_index(group_cols)["mem"]
test_df["best_observed_mem"] = test_df.set_index(group_cols).index.map(best_mem_map)
test_df["best_observed_time"] = test_df.groupby(group_cols)["stats_elapsed_time"].transform("min")

In [ ]:
group_cols = ["name", "nvar"]
best_idx = test_df.groupby(group_cols)["predicted_time"].idxmin()
model_time_map = test_df.loc[best_idx].set_index(group_cols)["stats_elapsed_time"]
best_mem_map = test_df.loc[best_idx].set_index(group_cols)["mem"]
test_df["best_predicted_time"] = test_df.set_index(group_cols).index.map(model_time_map)
test_df["best_predicted_mem"] = test_df.set_index(group_cols).index.map(best_mem_map)

In [ ]:
x = test_df.groupby(['name', 'nvar'])['best_predicted_mem'].first()
sns.histplot(x=x, discrete=True)
plt.xlim(0.5, x.max() + 0.5)
plt.xlabel("best_predicted_mem")
plt.ylabel("Count (unique problems)")
plt.title("Distribution of best_predicted_mem across unique problems")
plt.show()

In [ ]:
test_df.groupby(['name', 'nvar'])[['best_predicted_mem', 'best_observed_mem', 'predicted_time', 'stats_elapsed_time', 'best_observed_time', 'best_predicted_time']].first()

In [ ]:
x = test_df["best_predicted_mem"]
proportions = x.value_counts(normalize=True).sort_index()

print("Proportions of best_predicted_mem:")
print(proportions)

plt.figure(figsize=(10, 5))
sns.histplot(x=x, stat='percent', discrete=True, color='orange')
plt.xlabel("best_predicted_mem")
plt.ylabel("Percentage (%)")
plt.title("Distribution of best_predicted_mem")
plt.xlim(0.5, x.max() + 0.5)
plt.show()

### Mem evaluation metrics

In [ ]:
# How far off is the predicted mem from actual (in mem units)
mem_absolute_error = abs(test_df['best_predicted_mem'] - test_df['best_observed_mem'])
mame = np.mean(mem_absolute_error)
print(f"Mean Absolute Mem Error: {mame:.2f}")
print(f"Median Absolute Mem Error: {np.median(mem_absolute_error):.2f}")

In [ ]:
# How close are the predictions? 1.0 = perfect, 0.0 = very far
mem_match_ratio = np.minimum(test_df['best_predicted_mem'], 
                              test_df['best_observed_mem']) / \
                  np.maximum(test_df['best_predicted_mem'], 
                              test_df['best_observed_mem'])

mean_match_ratio = np.mean(mem_match_ratio)
print(f"Mean Mem Match Ratio: {mean_match_ratio:.3f}")
# 1.0 = perfect match
# 0.5 = off by 2x (e.g., pred=50, actual=100)
# 0.1 = off by 10x (e.g., pred=10, actual=100)

In [ ]:
group_cols = ['name', 'nvar']
unique_problems = test_df.groupby(group_cols).first().reset_index()

print(f"Total unique test problems: {len(unique_problems)}\n")

In [ ]:
print("="*60)
print("MEM PREDICTION ACCURACY")
print("="*60)

exact_match = (unique_problems['best_predicted_mem'] == unique_problems['best_observed_mem']).mean()
print(f"Exact match rate: {exact_match:.1%}")

within_5 = (abs(unique_problems['best_predicted_mem'] - unique_problems['best_observed_mem']) <= 5).mean()
within_10 = (abs(unique_problems['best_predicted_mem'] - unique_problems['best_observed_mem']) <= 10).mean()
within_20 = (abs(unique_problems['best_predicted_mem'] - unique_problems['best_observed_mem']) <= 20).mean()

print(f"\nWithin tolerance:")
print(f"  ± 5 mem units:  {within_5:.1%}")
print(f"  ± 10 mem units: {within_10:.1%}")
print(f"  ± 20 mem units: {within_20:.1%}")

### Baseline Comparison

In [ ]:
if 'baseline1_time' in test_df.columns:
    test_df = test_df.drop(columns='baseline1_time')
baseline1_values = df[df['mem'] == 5][['name', 'nvar', 'stats_elapsed_time']].drop_duplicates(['name', 'nvar'])
test_df = test_df.merge(
    baseline1_values.rename(columns={'stats_elapsed_time': 'baseline1_time'}), 
    on=['name', 'nvar'], 
    how='left'
)
test_df

In [ ]:
GLOBAL_BEST_MEM = 87
if 'baseline2_time' in test_df.columns:
    test_df = test_df.drop(columns='baseline2_time')
baseline2_values = df[df['mem'] == GLOBAL_BEST_MEM][['name', 'nvar', 'stats_elapsed_time']].drop_duplicates(['name', 'nvar'])
test_df = test_df.merge(
    baseline2_values.rename(columns={'stats_elapsed_time': 'baseline2_time'}), 
    on=['name', 'nvar'], 
    how='left'
)
test_df

In [ ]:
test_df = test_df.rename(columns={'best_observed_time' : 'oracle_time'})
test_df.head(41)

In [ ]:
# Model Regret (The memory value predicted by your RF)
test_df['model_regret'] = test_df['best_predicted_time'] - test_df['oracle_time']

# Baseline 1 Regret (Always using mem=5)
test_df['baseline1_regret'] = test_df['baseline1_time'] - test_df['oracle_time']

# Baseline 2 Regret (Always using the best fixed memory found in training)
test_df['baseline2_regret'] = test_df['baseline2_time'] - test_df['oracle_time']
test_df_unique = test_df.groupby(["name", "nvar"]).first()

test_df[test_df["model_regret"] == test_df["model_regret"].max()].head(1)

In [ ]:
print("\n" + "="*40)
print("AVERAGE REGRET (SECONDS ABOVE ORACLE)")
print("="*40)
print(test_df_unique[['model_regret', 'baseline1_regret', 'baseline2_regret']].mean().to_string())

print("\n" + "="*40)
print("REGRET QUANTILES (50% / 75% / 95%)")
print("="*40)
print(test_df_unique[['model_regret', 'baseline1_regret', 'baseline2_regret']].quantile([0.5, 0.75, 0.95]))

In [ ]:
# Calculate ratios per problem (Relative Slowdown)
test_df_unique['model_ratio'] = test_df_unique['best_predicted_time'] / test_df_unique['oracle_time']
test_df_unique['baseline1_ratio'] = test_df_unique['baseline1_time'] / test_df_unique['oracle_time']
test_df_unique['baseline2_ratio'] = test_df_unique['baseline2_time'] / test_df_unique['oracle_time']

# Print the comparison
print("\n" + "="*40)
print("PERFORMANCE RATIO (1.0 = PERFECTION)")
print("="*40)
cols = ['model_ratio', 'baseline1_ratio', 'baseline2_ratio']
print(test_df_unique[cols].quantile([0.5, 0.75, 0.95]))